# im gonna preen !!!
Initial stuff for BERTopic

### Initial Python Data Filtering for converting the data into Spark

Choose a data path below if you want to and put data into spark with filtering

In [1]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
  SparkSession.builder
  .appName("Phase2")
  .config("spark.driver.memory", "8g")
  .config("spark.executor.memory", "8g")
  .config("spark.sql.debug.maxToStringFields", "100")
  .getOrCreate()
)

review_df = spark.read.parquet("/home/cs179g/repo/data/clean_reviews.parquet")
row_count = review_df.count()
print("Number of rows: " + str(row_count))

26/11/20 20:48:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Number of rows: 1787844


In [4]:
review_df = review_df.withColumn('wordCount', F.size(F.split(F.col('review'), ' ')))
averageWord = review_df.select(F.mean('wordCount')).collect()
medianWord = review_df.select(F.median('wordCount')).collect()
print("Average Word Count: " + str(averageWord) + "\nMedian Word Count: " + str(medianWord))

Average Word Count: [Row(avg(wordCount)=13.328105248556362)]
Median Word Count: [Row(median(wordCount)=4.0)]


In [6]:
bigreview_df = review_df.filter(review_df.wordCount >= 5)
bigreview_df.count()

794497

In [9]:
bigreview_df = review_df.filter(review_df.wordCount >= 5)
small_df = bigreview_df.sample(withReplacement=False, fraction=0.4, seed=6)
row_count = small_df.count()
print("Number of rows: " + str(row_count))

Number of rows: 317345


In [11]:
small_df.show(10)

+----------------+-----------+------------------+----------------+------------------+-----------------------+-----------------+--------+-------------+--------------+-----------------+--------+--------------------+--------+--------+-----------+-------------------+---------------------------+-----------------+-----------------+------------+-------------------+--------+----+----------+---------+--------------------+--------------+--------------------+-------------------+---------+
|recommendationid|last_played|author_num_reviews|playtime_forever|playtime_at_review|playtime_last_two_weeks|          steamid|language|comment_count|steam_purchase|received_for_free|refunded|              review|voted_up|votes_up|votes_funny|weighted_vote_score|written_during_early_access|timestamp_created|timestamp_updated|has_hardware|adapter_description|cpu_name|  os|system_ram|vram_size|          clean_text|raw_word_count|post_review_playtime|timestamp_formatted|wordCount|
+----------------+-----------+----

Compile an array of reviews for the BERTopic model

In [12]:
from pyspark.sql.functions import collect_list
from pyspark.sql.functions import lit

reviewArr=(small_df.groupBy(lit(1)).agg(collect_list('review'))).first()['collect_list(review)']

### Initialize bertopic stuff

In [13]:
from bertopic import BERTopic
from bertopic import __init__
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from umap import UMAP

# umap_model = UMAP(n_neighbors=50, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = sentence_model.encode(reviewArr,
                                   show_progress_bar=True,
                                   normalize_embeddings=True,
                                   batch_size=256
                                   )

vectorizer_model = CountVectorizer(stop_words="english")
topic_model = BERTopic(
  vectorizer_model=vectorizer_model,
  low_memory=True,
  verbose=True,
  # umap_model=umap_model,
  min_topic_size=300,
  nr_topics=25
)
topics, probs = topic_model.fit_transform(reviewArr, embeddings)

/home/cs179g/miniconda3/envs/spark_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 1240/1240 [1:39:35<00:00,  4.82s/it] 
2026-11-20 22:50:58,221 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-11-20 23:02:59,470 - BERTopic - Dimensionality - Completed ✓
2026-11-20 23:02:59,506 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-11-20 23:04:20,090 - BERTopic - Cluster - Completed ✓
2026-11-20 23:04:20,092 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-11-20 23:04:27,956 - BERTopic - Representation - Completed ✓
2026-11-20 23:04:27,973 - BERTopic - Topic reduction - Reducing number of topics
2026-11-20 23:04:28,242 - BERTopic - Representation - Fine-tuning topics using representation m

In [14]:
topic_model.get_topic_info().head(25)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,96706,-1_game_vkcombagosi_play_good,"[game, vkcombagosi, play, good, like, fun, jus...","[Great game where u have to play as a team, if..."
1,0,155472,0_game_good_play_fun,"[game, good, play, fun, fps, best, csgo, cheat...",[I've played CS.GO for a thousand hours plus. ...
2,1,14338,1_russian_russians_learn_game,"[russian, russians, learn, game, speak, good, ...","[Good game, too many russians, good game, but ..."
3,2,8633,2_shooter_shoot_person_aim,"[shooter, shoot, person, aim, shooters, best, ...","[best first person shooter game ever, Best fir..."
4,3,6339,3_1010_r8_m8_88,"[1010, r8, m8, 88, gr8, ign, game, 1110, 10, p...","[Gr8 game would r8 8/8 m8, gr8 game m8, r8 8/8..."
5,4,6022,4_gotta_mom_gay_fast,"[gotta, mom, gay, fast, good, like, yeah, yes,...","[you are mom gay lol, im gay and i like it\n, ..."
6,5,5643,5_que_si_jogo_sa,"[que, si, jogo, sa, lmao, se, joc, la, nu, el]",[Sursa : http://steamcommunity.com/id/imkolok ...
7,6,4358,6_16_update_game_updates,"[16, update, game, updates, source, bugs, good...","[very shit with new update, Its good but the n..."
8,7,2823,7_knife_gamble_cases_gambling,"[knife, gamble, cases, gambling, case, open, a...","[good game, please give me knife, knife knife..."
9,8,2541,8_gud_veri_gg_ez,"[gud, veri, gg, ez, gam, gaem, hai, game, yang...","[2 gud 2 b 3, it's is is a gud, so gud i am so..."


In [22]:
import pandas as pd

pd.set_option('max_colwidth', None)
# pd.set_option('max_colwidth', 50)
topic_model.get_topic_info()["Representative_Docs"]

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       [Great game where u have to play as a team, if u get a good one., very good and fun game, Death is a great teacher. Failure in Counter-Strike: Global Offensive is, as it always has been for this series, the greatest way to learn where you should have gone, what you shouldn't have done, and how you could have done better. Counter-Strik

In [23]:
topic_model.get_topic_info()["Representation"]

0                [game, vkcombagosi, play, good, like, fun, just, time, dont, people]
1                     [game, good, play, fun, fps, best, csgo, cheaters, like, great]
2               [russian, russians, learn, game, speak, good, play, 1010, kids, dont]
3     [shooter, shoot, person, aim, shooters, best, shooting, terrorists, bomb, game]
4                                  [1010, r8, m8, 88, gr8, ign, game, 1110, 10, play]
5                             [gotta, mom, gay, fast, good, like, yeah, yes, did, im]
6                                      [que, si, jogo, sa, lmao, se, joc, la, nu, el]
7                 [16, update, game, updates, source, bugs, good, fix, lag, hitboxes]
8        [knife, gamble, cases, gambling, case, open, addiction, money, 1010, knives]
9                                [gud, veri, gg, ez, gam, gaem, hai, game, yang, ini]
10           [cs, 10, 16, offensive, global, csgo, counterstrike, valve, vac, source]
11                  [review, badge, doing, steam, ill,

In [39]:
embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
topic_model.save("/home/cs179g/repo/data/BERTopicModel", serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model)

In [25]:
import numpy as np
np.save("/home/cs179g/repo/data/embeddings.npy", embeddings)

In [ ]:
sentences = ["Why so many smurfs"]
embeddings = sentence_model.encode(sentences)
topics, probs = topic_model.transform(sentences, embeddings)

2026-11-21 00:08:59,069 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-11-21 00:08:59,111 - BERTopic - Dimensionality - Completed ✓
2026-11-21 00:08:59,112 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-11-21 00:08:59,117 - BERTopic - Cluster - Completed ✓


In [36]:
print(topics)

([np.int64(12)], array([0.85192383]))


In [37]:
mergeTopics = [0, 5, 23]
topic_model.merge_topics(reviewArr, mergeTopics)

In [38]:
topic_model.get_topic_info().head(23)

Topic   Count                                   Name  \
0      -1   96706          -1_game_vkcombagosi_play_good   
1       0  161485                   0_game_good_play_fun   
2       1   14338          1_russian_russians_game_learn   
3       2    8633             2_shooter_shoot_person_aim   
4       3    6339                        3_1010_r8_m8_88   
5       4    6022                   4_gotta_mom_gay_fast   
6       5    4358               5_16_update_game_updates   
7       6    2823          6_knife_gamble_cases_gambling   
8       7    2541                       7_gud_veri_gg_ez   
9       8    2317                   8_cs_10_16_offensive   
10      9    1782             9_review_badge_doing_steam   
11     10    1689                10_cyka_blyat_rush_syka   
12     11    1496           11_smurfs_smurf_hackers_game   
13     12    1256           12_valorant_better_game_play   
14     13    1150            13_bugs_average_grind_story   
15     14    1017          14_chickens_chicken_kill_1010   
16     15     659               15_surf_surfing_fun_bhop   
17     16     637            16_pew_pewpew_pow_pewpewpew   
18     17     477                  17_volvo_fix_pls_game   
19     18     464                 18_awp_1010_awps_awped   
20     19     416          19_anticheat_fix_better_needs   
21     20     370  20_relationship_lovehate_hate_abusive   
22     21     370              21_h1_strike_game_spoiler   

                                                                     Representation  \
0              [game, vkcombagosi, play, good, like, fun, just, time, dont, people]   
1                   [game, good, play, fun, fps, best, csgo, like, cheaters, great]   
2             [russian, russians, game, learn, speak, good, play, 1010, kids, dont]   
3   [shooter, shoot, person, aim, best, shooters, shooting, terrorists, bomb, game]   
4                                [1010, r8, m8, 88, gr8, ign, game, 1110, 10, play]   
5                           [gotta, mom, gay, fast, good, like, yeah, yes, did, im]   
6               [16, update, game, updates, source, good, bugs, fix, lag, hitboxes]   
7      [knife, gamble, cases, gambling, case, open, addiction, money, 1010, knives]   
8                              [gud, veri, gg, ez, gam, gaem, hai, game, yang, ini]   
9          [cs, 10, 16, offensive, global, csgo, counterstrike, valve, vac, source]   
10                [review, badge, doing, steam, ill, yeah, just, im, reviews, game]   
11                [cyka, blyat, rush, syka, blyad, suka, 1010, stop, game, rushing]   
12           [smurfs, smurf, hackers, game, fun, good, smurfing, play, 1010, great]   
13             [valorant, better, game, play, like, csgo, just, good, fun, instead]   
14              [bugs, average, grind, story, long, price, bad, spare, paint, easy]   
15        [chickens, chicken, kill, 1010, hog, bloody, rider, promise, touch, mess]   
16              [surf, surfing, fun, bhop, play, game, really, like, servers, good]   
17                [pew, pewpew, pow, pewpewpew, bang, boom, gun, guns, shoot, haha]   
18               [volvo, fix, pls, game, cheaters, plz, volovo, thanks, good, shit]   
19                    [awp, 1010, awps, awped, awpd, shot, game, kill, awper, nerf]   
20              [anticheat, fix, better, needs, game, good, valve, bad, shit, need]   
21       [relationship, lovehate, hate, abusive, love, like, ex, game, come, toxic]   
22             [h1, strike, game, spoiler, list, strikewill, gameh1, h1i, play, h2]   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [40]:
loaded_model = BERTopic.load("/home/cs179g/repo/data/BERTopicModel")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6724.88it/s]


In [42]:
sentences = ["Why so many smurfs"]
embeddings = sentence_model.encode(sentences)
topics, probs = loaded_model.transform(sentences, embeddings)
print(topics[0])

2026-11-21 00:48:15,037 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


11
